In [1]:
import import_ipynb
from yahoo import getSP500_data
from cpi import get_cpi
from ffer import get_dff
from unemployment import get_unrate

In [2]:
cpi = get_cpi()
fedfunds = get_dff()
unrate = get_unrate()
sp500 = getSP500_data()

In [5]:
keys = ["Month", "Year"]

def one_row_per_month_year(df):
    # Keep one record per Month/Year to avoid repeated rows after merge.
    if "Date" in df.columns:
        return df.sort_values("Date").drop_duplicates(subset=keys, keep="last")
    if "observation_date" in df.columns:
        return df.sort_values("observation_date").drop_duplicates(subset=keys, keep="last")
    return df.drop_duplicates(subset=keys, keep="last")

cpi_u = one_row_per_month_year(cpi)
fedfunds_u = one_row_per_month_year(fedfunds)
unrate_u = one_row_per_month_year(unrate)
sp500_u = one_row_per_month_year(sp500)

merged_df = (
    sp500_u.merge(cpi_u, on=keys, how="inner")
           .merge(fedfunds_u, on=keys, how="inner")
           .merge(unrate_u, on=keys, how="inner")
)

# Drop date columns that appear from source DataFrames.
drop_cols = [
    c for c in merged_df.columns
    if c.lower() in {"date", "observation_date_x", "observation_date", "observation_date_y"}
]
merged_df = merged_df.drop(columns=drop_cols, errors="ignore")
merged_df = merged_df.drop(columns=["First_Closing_Price", "Final_Closing_Price"], errors="ignore")
# Keep only the final columns in the requested order.

merged_df.head()

,Monthly_Increase,Month,Year,cpi_pct,fedfunds_rate,unemployment_rate
0,-3.870003,1,1968,3.651861,4.75,3.7
1,-3.199997,2,1968,3.673819,4.75,3.8
2,1.089996,3,1968,4.142164,5.25,3.7
3,4.979996,4,1968,4.155828,6.25,3.5
4,0.709999,5,1968,4.088245,6.13,3.5


In [6]:
from pathlib import Path

download_path = Path.cwd().parent / "merged.csv"
merged_df.to_csv(download_path, index=False)